# N078 · Trie与回溯的组合

**目标：** 用词典前缀剪枝减少无效路径搜索。

**先修：** N067, N077。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** Trie状态；通配符分支；网格DFS；结果去重；访问恢复。

## 从问题到算法

Trie与回溯结合时，状态同时包含网格位置、当前路径已访问格和词典前缀。不存在相应前缀就立即剪枝，比对每个单词单独搜索更早共享失败信息。找到词后移除终止标记以去重；没有孩子也没有终止标记的分支可删除，但棋盘必须恢复。通配符`.`匹配恰好一个字符，不是任意长度。

## 最小实现

**本章接口：** `WordDictionary`、`find_words(board, words)`

In [1]:
class TrieNode:
    def __init__(self): self.children={}; self.terminal=False

class Trie:
    def __init__(self): self.root=TrieNode()
    def insert(self,word):
        node=self.root
        for c in word:
            if c not in node.children: node.children[c]=TrieNode()
            node=node.children[c]
        node.terminal=True
    def _walk(self,text):
        node=self.root
        for c in text:
            if c not in node.children: return None
            node=node.children[c]
        return node
    def search(self,word):
        node=self._walk(word); return node is not None and node.terminal
    def startsWith(self,prefix): return self._walk(prefix) is not None

class WordDictionary:
    def __init__(self): self.trie=Trie()
    def addWord(self,word): self.trie.insert(word)
    def search(self,word):
        def visit(node,i):
            if i==len(word): return node.terminal
            if word[i]=='.': return any(visit(child,i+1) for child in node.children.values())
            child=node.children.get(word[i]); return child is not None and visit(child,i+1)
        return visit(self.trie.root,0)

def find_words(board,words):
    if not board or not board[0]: return []
    root={}
    for word in words:
        if not word: raise ValueError('nonempty dictionary words required')
        node=root
        for c in word: node=node.setdefault(c,{})
        node[None]=word
    rows,cols=len(board),len(board[0]); result=[]
    def visit(r,c,parent):
        char=board[r][c]
        if char is None or char not in parent: return
        node=parent[char]; found=node.pop(None,None)
        if found is not None: result.append(found)
        board[r][c]=None
        try:
            for dr,dc in [(1,0),(-1,0),(0,1),(0,-1)]:
                nr,nc=r+dr,c+dc
                if 0<=nr<rows and 0<=nc<cols: visit(nr,nc,node)
        finally: board[r][c]=char
        if not node: del parent[char]
    for r in range(rows):
        for c in range(cols): visit(r,c,root)
    return result

## 正确性、前提与复杂度

前缀不存在时任何继续延伸都不可能变成字典词，剪枝安全。找到终止词不等于终止整个分支，因为该词可能仍是其他词的前缀。

**代价：** 词典构建O(总词长)；网格搜索最坏仍指数，粗界O(RC·4ᴸ)，Trie共享与剪枝改善实际工作；额外Trie空间O(总词长)、路径栈O(L)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

board=[list('oaan'),list('etae'),list('ihkr'),list('iflv')]
show_table(['行','字符'],list(enumerate(board)))
show_table(['词典','找到的词'],[(['oath','pea','eat','rain'],sorted(find_words(board,['oath','pea','eat','rain'])))])

行,字符
0,"['o', 'a', 'a', 'n']"
1,"['e', 't', 'a', 'e']"
2,"['i', 'h', 'k', 'r']"
3,"['i', 'f', 'l', 'v']"


词典,找到的词
"['oath', 'pea', 'eat', 'rain']","['eat', 'oath']"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from copy import deepcopy
wd=WordDictionary()
for w in ['bad','dad','mad']: wd.addWord(w)
assert wd.search('.a.') and wd.search('b..') and not wd.search('..')
b=[list('oaan'),list('etae'),list('ihkr'),list('iflv')]; before=deepcopy(b)
assert set(find_words(b,['oath','pea','eat','rain','eat']))=={'oath','eat'} and b==before
def slow_exist(board,word):
    def dfs(r,c,i,used):
        if board[r][c]!=word[i]: return False
        if i+1==len(word): return True
        for nr,nc in [(r+1,c),(r-1,c),(r,c+1),(r,c-1)]:
            if 0<=nr<len(board) and 0<=nc<len(board[0]) and (nr,nc) not in used:
                if dfs(nr,nc,i+1,used|{(nr,nc)}): return True
        return False
    return any(dfs(r,c,0,{(r,c)}) for r in range(len(board)) for c in range(len(board[0])))
from itertools import product
words=[''.join(x) for n in range(1,4) for x in product('ab',repeat=n)]
for values in product('ab',repeat=4):
    b=[list(values[:2]),list(values[2:])]; before=deepcopy(b)
    assert set(find_words(b,words))=={w for w in words if slow_exist(b,w)} and b==before
print('N078: 所有本章断言通过')

N078: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 对比逐词搜索和共享Trie搜索。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 处理词典中重复词与前缀词。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 211. Design Add and Search Words Data Structure（canonical）
- 212. Word Search II（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。